In [ ]:
import pandas as pd
import os

seasons = [
    "18-19", "19-20", "20-21",
    "21-22", "22-23", "23-24",
    "24-25", "25-26"
]

dfs = []

for division in ["D1", "D2"]:
    for season in seasons:
        file = f"rawdata/{division} {season}.csv"

        temp = pd.read_csv(file)

        temp["Season"] = season
        temp["Division"] = division

        dfs.append(temp)

df = pd.concat(dfs, ignore_index=True)

# Parse dates immediately so every later sort/group is chronological.
# (Matchday used to be built from the raw dd/mm/yyyy strings, which sorts by day number.)
df["Date"] = pd.to_datetime(df["Date"], format="%d/%m/%Y")

# Normalise team names to plain ASCII ("Preußen Münster" -> "Preussen Munster")
# so they are consistent with Koln, Nurnberg, Wurzburger Kickers, etc.
import unicodedata

def ascii_name(name):
    name = name.replace("ß", "ss")
    return unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode()

for col in ["HomeTeam", "AwayTeam"]:
    df[col] = df[col].map(ascii_name)

print("Dataset shape:", df.shape)
print(df[["Season", "Division", "HomeTeam", "AwayTeam", "FTR"]].head())

In [ ]:
df_d1 = df[df["Division"] == "D1"].copy()
df_d2 = df[df["Division"] == "D2"].copy()

df_d1.reset_index(drop=True, inplace=True)
df_d2.reset_index(drop=True, inplace=True)

print("Bundesliga 1:", df_d1.shape)
print("Bundesliga 2:", df_d2.shape)

In [ ]:
# Add matchday feature (1–34)
# Calculate separately for each season and division.
# Dates are already datetime (cell 0), so sorting is chronological.
# Matchday = position in the season's date order // 9 matches per round.
# Rescheduled games make this approximate for a few postponed matches.

def add_matchday(df):
    df = df.copy()

    df = df.sort_values(
        ["Season", "Date", "HomeTeam"]
    ).reset_index(drop=True)

    df["Matchday"] = (
        df.groupby("Season").cumcount() // 9 + 1
    )

    return df


df_d1 = add_matchday(df_d1)
df_d2 = add_matchday(df_d2)

# Sanity check: Matchday must rise with date within every season
for name, data in [("Bundesliga 1", df_d1), ("Bundesliga 2", df_d2)]:
    for season, g in data.groupby("Season"):
        assert g["Matchday"].is_monotonic_increasing, f"{name} {season}: Matchday not chronological"
        assert g["Matchday"].max() == 34, f"{name} {season}: expected 34 matchdays"

display(df_d1[["Season", "Date", "HomeTeam", "AwayTeam", "Matchday"]].head(15))


In [ ]:
# Correct missing match statistics for Union Berlin vs Bochum (December 14, 2024).
# The match ended 1-1 on the field but was officially awarded 0-2 to Bochum.
# Keep the official result while restoring actual match statistics.

match = (
    (df_d1["Season"] == "24-25") &
    (df_d1["HomeTeam"] == "Union Berlin") &
    (df_d1["AwayTeam"] == "Bochum")
)

corrections = {
    "HTHG": 1, "HTAG": 1, "HTR": "D",
    "HS": 22, "AS": 13,
    "HST": 4, "AST": 3,
    "HC": 7, "AC": 6,
    "HF": 11, "AF": 9,
    "HY": 2, "AY": 2,
    "HR": 0, "AR": 1
}

assert match.sum() == 1, "Expected exactly one matching game"

for col, value in corrections.items():
    df_d1.loc[match, col] = value

print("Corrected match:")
display(df_d1.loc[match])

In [ ]:
keep_cols = [
    "Season", "Division", "Date",
    "HomeTeam", "AwayTeam",
    "FTHG", "FTAG", "FTR",
    "HTHG", "HTAG", "HTR",
    "HS", "AS", "HST", "AST",
    "HC", "AC", "HF", "AF",
    "HY", "AY", "HR", "AR", 'Matchday',
]

df_d1 = df_d1[keep_cols].copy()
df_d2 = df_d2[keep_cols].copy()

print("Bundesliga 1:", df_d1.shape)
print("Bundesliga 2:", df_d2.shape)

print("\nMissing values in Bundesliga 1:")
print(df_d1.isnull().sum())

print("\nMissing values in Bundesliga 2:")
print(df_d2.isnull().sum())

In [ ]:
# Identify Bundesliga 1 matches with missing statistics

missing_matches = df_d1[df_d1.isnull().any(axis=1)]

print(missing_matches.to_string(index=False))

In [ ]:
# Convert match dates to datetime and sort both divisions chronologically

for data in [df_d1, df_d2]:
    # Date was already parsed in cell 0; this keeps the cell safe to re-run
    data["Date"] = pd.to_datetime(data["Date"])

    data.sort_values(
        by=["Date", "Season"],
        inplace=True
    )

    data.reset_index(drop=True, inplace=True)

print("Bundesliga 1:")
print(df_d1[["Date", "HomeTeam", "AwayTeam"]].head())

print("\nBundesliga 2:")
print(df_d2[["Date", "HomeTeam", "AwayTeam"]].head())

In [ ]:
# Check unique team names across both divisions and all seasons

teams_d1 = sorted(
    set(df_d1["HomeTeam"]) | set(df_d1["AwayTeam"])
)

teams_d2 = sorted(
    set(df_d2["HomeTeam"]) | set(df_d2["AwayTeam"])
)

print("Bundesliga 1 teams:", len(teams_d1))
print(teams_d1)

print("\nBundesliga 2 teams:", len(teams_d2))
print(teams_d2)

In [ ]:
# Check for duplicate matches using season, date, and team names

match_cols = ["Season", "Date", "HomeTeam", "AwayTeam"]

duplicates_d1 = df_d1[df_d1.duplicated(subset=match_cols, keep=False)]
duplicates_d2 = df_d2[df_d2.duplicated(subset=match_cols, keep=False)]

print("Bundesliga 1 duplicate matches:", len(duplicates_d1))
print("Bundesliga 2 duplicate matches:", len(duplicates_d2))

display(duplicates_d1)
display(duplicates_d2)

In [ ]:
# Calculate points earned by home and away teams in each match

points_map = {
    "H": (3, 0),
    "D": (1, 1),
    "A": (0, 3)
}

for data in [df_d1, df_d2]:
    data["HomePoints"] = data["FTR"].map(lambda x: points_map[x][0])
    data["AwayPoints"] = data["FTR"].map(lambda x: points_map[x][1])

print("Bundesliga 1:")
display(df_d1[["HomeTeam", "AwayTeam", "FTR", "HomePoints", "AwayPoints"]].head())

print("Bundesliga 2:")
display(df_d2[["HomeTeam", "AwayTeam", "FTR", "HomePoints", "AwayPoints"]].head())

In [ ]:
# Create a team-level match history for both Bundesliga divisions.
# Previously, each row represented one match with two teams (home and away).
# This function separates each match into two rows, one from each team's perspective.
# It records each team's points, goals, shots, corners, fouls, and cards,
# including both their own statistics ("For") and their opponent's ("Against").
# Finally, it combines the home/away records and sorts them by team and date.


def create_team_history(data):
    stat_cols = {
        "Goals": ("FTHG", "FTAG"),
        "Shots": ("HS", "AS"),
        "ShotsOnTarget": ("HST", "AST"),
        "Corners": ("HC", "AC"),
        "Fouls": ("HF", "AF"),
        "YellowCards": ("HY", "AY"),
        "RedCards": ("HR", "AR")
    }

    histories = []

    for venue in ["Home", "Away"]:
        temp = data[[
            "Season", "Division", "Date",
            "HomeTeam", "AwayTeam", "HomePoints", "AwayPoints"
        ] + [col for pair in stat_cols.values() for col in pair]].copy()

        temp["Team"] = temp["HomeTeam"] if venue == "Home" else temp["AwayTeam"]
        temp["Opponent"] = temp["AwayTeam"] if venue == "Home" else temp["HomeTeam"]
        temp["Points"] = temp["HomePoints"] if venue == "Home" else temp["AwayPoints"]
        temp["Venue"] = venue

        for stat, (home_col, away_col) in stat_cols.items():
            temp[f"{stat}For"] = temp[home_col] if venue == "Home" else temp[away_col]
            temp[f"{stat}Against"] = temp[away_col] if venue == "Home" else temp[home_col]

        keep = ["Season", "Division", "Date", "Team", "Opponent", "Points", "Venue"]
        keep += [f"{stat}{side}" for stat in stat_cols for side in ["For", "Against"]]

        histories.append(temp[keep])

    history = pd.concat(histories, ignore_index=True)
    history = history.sort_values(["Team", "Date"]).reset_index(drop=True)

    return history


history_d1 = create_team_history(df_d1)
history_d2 = create_team_history(df_d2)

print("Bundesliga 1:", history_d1.shape)
print("Bundesliga 2:", history_d2.shape)

display(history_d1.head())

In [ ]:
# Calculate each team's statistics from their previous 5 matches.
# Group by team and season so rolling statistics reset each season.
# Use shift(1) to exclude the current match and prevent data leakage.
# Require 5 previous matches before calculating any rolling features.

def add_rolling_features(history):
    history = history.copy()

    stat_cols = [
        "Points",
        "GoalsFor", "GoalsAgainst",
        "ShotsFor", "ShotsAgainst",
        "ShotsOnTargetFor", "ShotsOnTargetAgainst",
        "CornersFor", "CornersAgainst",
        "FoulsFor", "FoulsAgainst",
        "YellowCardsFor", "YellowCardsAgainst",
        "RedCardsFor", "RedCardsAgainst"
    ]

    for col in stat_cols:
        grouped = history.groupby(["Season", "Team"])[col]

        if col == "Points":
            history["Last5Points"] = grouped.transform(
                lambda x: x.shift(1).rolling(5, min_periods=5).sum()
            )
        else:
            history[f"Last5Avg{col}"] = grouped.transform(
                lambda x: x.shift(1).rolling(5, min_periods=5).mean()
            )

    return history


history_d1 = add_rolling_features(history_d1)
history_d2 = add_rolling_features(history_d2)

display(
    history_d1[
        ["Season", "Date", "Team", "Points",
         "Last5Points", "Last5AvgGoalsFor", "Last5AvgGoalsAgainst"]
    ].head(10)
)

In [ ]:
# Merge each team's previous 5-game statistics into the original match data.
# Create separate Home_ and Away_ features for each match.
# Match using season, date, and team names to attach the correct history.

def merge_rolling_features(data, history):
    rolling_cols = [
        col for col in history.columns
        if col.startswith("Last5")
    ]

    home = history[history["Venue"] == "Home"][
        ["Season", "Date", "Team", "Opponent"] + rolling_cols
    ].copy()

    away = history[history["Venue"] == "Away"][
        ["Season", "Date", "Team", "Opponent"] + rolling_cols
    ].copy()

    home = home.rename(columns={
        "Team": "HomeTeam",
        "Opponent": "AwayTeam",
        **{col: f"Home_{col}" for col in rolling_cols}
    })

    away = away.rename(columns={
        "Team": "AwayTeam",
        "Opponent": "HomeTeam",
        **{col: f"Away_{col}" for col in rolling_cols}
    })

    match_keys = ["Season", "Date", "HomeTeam", "AwayTeam"]

    data = data.merge(home, on=match_keys, how="left", validate="one_to_one")
    data = data.merge(away, on=match_keys, how="left", validate="one_to_one")

    return data


df_d1 = merge_rolling_features(df_d1, history_d1)
df_d2 = merge_rolling_features(df_d2, history_d2)

print("Bundesliga 1:", df_d1.shape)
print("Bundesliga 2:", df_d2.shape)

display(df_d1[
    ["Date", "HomeTeam", "AwayTeam",
     "Home_Last5Points", "Away_Last5Points", "FTR"]
].head(15))

In [ ]:
# Verify that rolling features were merged correctly for both divisions.
# Count matches with complete five-game features and inspect sample matches.

for name, data in [("Bundesliga 1", df_d1), ("Bundesliga 2", df_d2)]:
    valid = data.dropna(
        subset=["Home_Last5Points", "Away_Last5Points"]
    )

    print(f"{name}:")
    print("Matches with both teams having 5 previous games:", len(valid))
    print("Matches without complete 5-game points:", len(data) - len(valid))

    display(valid[
        ["Season", "Date", "HomeTeam", "AwayTeam",
         "Home_Last5Points", "Away_Last5Points", "FTR"]
    ].head())

In [ ]:
# Calculate total points earned by each team in each season.
# Combine both divisions so we can track teams across promotions and relegations.
# This creates a season-level summary for future historical points features.

all_history = pd.concat(
    [history_d1, history_d2],
    ignore_index=True
)

season_points = (
    all_history
    .groupby(["Season", "Division", "Team"], as_index=False)
    .agg(
        SeasonPoints=("Points", "sum"),
        MatchesPlayed=("Points", "count")
    )
)

season_points["PointsPerGame"] = (
    season_points["SeasonPoints"] / season_points["MatchesPlayed"]
)

display(season_points.head(10))

In [ ]:
# Calculate historical team performance using up to 5 previous seasons.
# Use shift(1) to exclude the current season and prevent data leakage.
# Group by team across both divisions to account for promotion and relegation.
# Track how many previous seasons are available for each team.

season_order = {season: i for i, season in enumerate(seasons)}

season_points = season_points.copy()
season_points["SeasonOrder"] = season_points["Season"].map(season_order)

season_points = season_points.sort_values(
    ["Team", "SeasonOrder"]
).reset_index(drop=True)

grouped = season_points.groupby("Team")

season_points["Prev5SeasonAvgPPG"] = grouped["PointsPerGame"].transform(
    lambda x: x.shift(1).rolling(5, min_periods=1).mean()
)

season_points["Prev5SeasonAvgPoints"] = grouped["SeasonPoints"].transform(
    lambda x: x.shift(1).rolling(5, min_periods=1).mean()
)

season_points["PrevSeasonsAvailable"] = grouped["Season"].transform(
    lambda x: x.shift(1).rolling(5, min_periods=1).count()
)

display(
    season_points[
        ["Season", "Division", "Team", "SeasonPoints",
         "Prev5SeasonAvgPPG", "Prev5SeasonAvgPoints",
         "PrevSeasonsAvailable"]
    ].query("Team == 'Bayern Munich'")
)

In [ ]:
# Merge previous 5-season performance into each match.
# Match teams by season and name, accounting for promotion/relegation.
# Add separate Home_ and Away_ historical features for modeling.

def merge_season_features(data, season_points):
    feature_cols = [
        "Prev5SeasonAvgPPG",
        "Prev5SeasonAvgPoints",
        "PrevSeasonsAvailable"
    ]

    home = season_points[
        ["Season", "Team"] + feature_cols
    ].rename(columns={
        "Team": "HomeTeam",
        **{col: f"Home_{col}" for col in feature_cols}
    })

    away = season_points[
        ["Season", "Team"] + feature_cols
    ].rename(columns={
        "Team": "AwayTeam",
        **{col: f"Away_{col}" for col in feature_cols}
    })

    data = data.merge(
        home,
        on=["Season", "HomeTeam"],
        how="left",
        validate="many_to_one"
    )

    data = data.merge(
        away,
        on=["Season", "AwayTeam"],
        how="left",
        validate="many_to_one"
    )

    return data


df_d1 = merge_season_features(df_d1, season_points)
df_d2 = merge_season_features(df_d2, season_points)

print("Bundesliga 1:", df_d1.shape)
print("Bundesliga 2:", df_d2.shape)

display(df_d1[
    ["Season", "HomeTeam", "AwayTeam",
     "Home_Prev5SeasonAvgPPG", "Away_Prev5SeasonAvgPPG"]
].tail(10))

In [ ]:
# Calculate head-to-head records from the previous 5 meetings.
# Combine both divisions to include meetings before promotion/relegation.
# Use only earlier match dates to prevent data leakage.
# Count wins for the current home team, wins for the current away team, and draws.

all_matches = pd.concat([df_d1, df_d2], ignore_index=True)
all_matches = all_matches.sort_values("Date").copy()

h2h_records = {}

for idx, match in all_matches.iterrows():
    home = match["HomeTeam"]
    away = match["AwayTeam"]

    # Use a sorted pair so the matchup is the same regardless of venue
    pair = tuple(sorted([home, away]))

    previous = h2h_records.get(pair, [])

    # Only use meetings strictly before the current date
    previous_five = [m for m in previous if m[0] < match["Date"]][-5:]

    all_matches.loc[idx, "H2H_HomeWins"] = sum(
        m[1] == home for m in previous_five
    )

    all_matches.loc[idx, "H2H_AwayWins"] = sum(
        m[1] == away for m in previous_five
    )

    all_matches.loc[idx, "H2H_Draws"] = sum(
        m[1] is None for m in previous_five
    )

    all_matches.loc[idx, "H2H_Matches"] = len(previous_five)

    # Record the winner after calculating historical features
    if match["FTR"] == "H":
        winner = home
    elif match["FTR"] == "A":
        winner = away
    else:
        winner = None

    h2h_records.setdefault(pair, []).append((match["Date"], winner))


df_d1 = all_matches[all_matches["Division"] == "D1"].copy()
df_d2 = all_matches[all_matches["Division"] == "D2"].copy()

df_d1.reset_index(drop=True, inplace=True)
df_d2.reset_index(drop=True, inplace=True)

print("Bundesliga 1:", df_d1.shape)
print("Bundesliga 2:", df_d2.shape)

display(df_d1[
    ["Date", "HomeTeam", "AwayTeam",
     "H2H_HomeWins", "H2H_AwayWins", "H2H_Draws", "H2H_Matches"]
].tail(10))

In [ ]:
# Calculate the number of days since each team's previous match.
# Use both Bundesliga divisions to track teams across promotion/relegation.
# Sort by team and date, then use groupby() and diff() to calculate rest days.
# Merge the results back into the match datasets for home and away teams.

def add_rest_days(df_d1, df_d2):
    all_matches = pd.concat([df_d1, df_d2], ignore_index=True)

    home = all_matches[
        ["Season", "Date", "HomeTeam", "AwayTeam"]
    ].copy()
    home["Team"] = home["HomeTeam"]
    home["Venue"] = "Home"

    away = all_matches[
        ["Season", "Date", "HomeTeam", "AwayTeam"]
    ].copy()
    away["Team"] = away["AwayTeam"]
    away["Venue"] = "Away"

    history = pd.concat([home, away], ignore_index=True)
    history = history.sort_values(["Team", "Date"])

    history["DaysSinceLastMatch"] = (
        history.groupby("Team")["Date"]
        .diff()
        .dt.days
    )

    keys = ["Season", "Date", "HomeTeam", "AwayTeam"]

    home_rest = history[history["Venue"] == "Home"][
        keys + ["DaysSinceLastMatch"]
    ].rename(columns={
        "DaysSinceLastMatch": "Home_RestDays"
    })

    away_rest = history[history["Venue"] == "Away"][
        keys + ["DaysSinceLastMatch"]
    ].rename(columns={
        "DaysSinceLastMatch": "Away_RestDays"
    })

    all_matches = all_matches.merge(
        home_rest, on=keys, how="left", validate="one_to_one"
    )

    all_matches = all_matches.merge(
        away_rest, on=keys, how="left", validate="one_to_one"
    )

    return (
        all_matches[all_matches["Division"] == "D1"].reset_index(drop=True),
        all_matches[all_matches["Division"] == "D2"].reset_index(drop=True)
    )


df_d1, df_d2 = add_rest_days(df_d1, df_d2)

print("Bundesliga 1:", df_d1.shape)
print("Bundesliga 2:", df_d2.shape)

display(df_d1[
    ["Date", "HomeTeam", "AwayTeam", "Home_RestDays", "Away_RestDays"]
].head(15))

In [ ]:
# Check whether the original datasets contain stadium or attendance information.
# Search all original column names for relevant keywords before using external data.

keywords = ["stadium", "capacity", "attendance", "venue", "crowd"]

stadium_cols = [
    col for col in df.columns
    if any(word in col.lower() for word in keywords)
]

print("Potential stadium-related columns:", stadium_cols)

In [ ]:
# Check missing values in all engineered features for both Bundesliga divisions.
# Identify which features have missing data and how many matches are affected.
# This helps us decide which rows to remove or which values to handle.

for name, data in [("Bundesliga 1", df_d1), ("Bundesliga 2", df_d2)]:

    feature_cols = [
        col for col in data.columns
        if col.startswith(("Home_Last5", "Away_Last5",
                           "Home_Prev5", "Away_Prev5",
                           "Home_PrevSeasons", "Away_PrevSeasons",
                           "H2H_", "Home_Rest", "Away_Rest"))
    ]

    missing = data[feature_cols].isnull().sum()
    missing = missing[missing > 0].sort_values(ascending=False)

    print(f"\n{name}")
    print("Total matches:", len(data))
    print("Features with missing values:")
    print(missing.to_string())

    print("\nMatches with all engineered features available:",
          data[feature_cols].notnull().all(axis=1).sum())

In [ ]:
# Check how many matches have complete previous 5-game statistics.
# Do not remove rows yet; only calculate how many would be retained.
# Historical season averages and rest days will be handled separately.

for name, data in [("Bundesliga 1", df_d1), ("Bundesliga 2", df_d2)]:

    rolling_cols = [
        col for col in data.columns
        if col.startswith(("Home_Last5", "Away_Last5"))
    ]

    complete_rolling = data[rolling_cols].notna().all(axis=1)

    print(f"\n{name}")
    print("Total matches:", len(data))
    print("Matches with complete rolling features:", complete_rolling.sum())
    print("Matches removed:", (~complete_rolling).sum())

    print("\nRemaining missing values:")
    remaining = data.loc[complete_rolling]
    missing = remaining.isna().sum()
    print(missing[missing > 0].to_string())

In [ ]:
# Create final working datasets using matches with complete previous 5-game features.
# Exclude matches without enough historical match statistics.
# Preserve missing previous-season averages for later model preprocessing.
# Keep the original DataFrames unchanged for validation.

rolling_cols_d1 = [
    col for col in df_d1.columns
    if col.startswith(("Home_Last5", "Away_Last5"))
]

rolling_cols_d2 = [
    col for col in df_d2.columns
    if col.startswith(("Home_Last5", "Away_Last5"))
]

final_d1 = df_d1.dropna(subset=rolling_cols_d1).copy()
final_d2 = df_d2.dropna(subset=rolling_cols_d2).copy()

final_d1.reset_index(drop=True, inplace=True)
final_d2.reset_index(drop=True, inplace=True)

print("Final Bundesliga 1:", final_d1.shape)
print("Final Bundesliga 2:", final_d2.shape)

In [ ]:
# Validate both final datasets before exporting.
# Check duplicates, target values, rolling features, and historical feature ranges.
# Identify any remaining missing values for teammates to handle during modeling.

for name, data in [("Bundesliga 1", final_d1), ("Bundesliga 2", final_d2)]:

    print(f"\n{name}")
    print("Dataset shape:", data.shape)

    # Check duplicate matches
    duplicates = data.duplicated(
        subset=["Season", "Date", "HomeTeam", "AwayTeam"]
    ).sum()
    print("Duplicate matches:", duplicates)

    # Check match outcomes
    print("Match outcomes:", data["FTR"].value_counts().to_dict())
    print("Invalid outcomes:", (~data["FTR"].isin(["H", "D", "A"])).sum())

    # Check rolling features
    rolling_cols = [
        col for col in data.columns
        if col.startswith(("Home_Last5", "Away_Last5"))
    ]
    print("Missing rolling values:", data[rolling_cols].isna().sum().sum())

    # Check points are between 0 and 15
    for col in ["Home_Last5Points", "Away_Last5Points"]:
        invalid = (~data[col].between(0, 15)).sum()
        print(f"Invalid {col}:", invalid)

    # Check H2H counts are consistent
    h2h_total = (
        data["H2H_HomeWins"]
        + data["H2H_AwayWins"]
        + data["H2H_Draws"]
    )
    print("Invalid H2H totals:",
          (h2h_total != data["H2H_Matches"]).sum())

    # Matchday should be chronological within each season
    bad_md = [s for s, g in data.sort_values("Date").groupby("Season")
              if not g["Matchday"].is_monotonic_increasing]
    print("Seasons with non-chronological Matchday:", bad_md)

    # Team names should be ASCII only
    names = set(data["HomeTeam"]) | set(data["AwayTeam"])
    print("Non-ASCII team names:", [n for n in names if not n.isascii()])

    # Display remaining missing columns
    missing = data.isna().sum()
    print("\nRemaining missing values:")
    print(missing[missing > 0].to_string())

In [ ]:
# Summarize all completed feature engineering for both Bundesliga divisions.
# Display dataset sizes, engineered feature categories, and example matches.
# This provides a clear overview of the final datasets for teammates.

feature_groups = {
    "Previous 5-Game Performance": [
        col for col in final_d1.columns
        if col.startswith(("Home_Last5", "Away_Last5"))
    ],
    "Previous 5-Season Performance": [
        col for col in final_d1.columns
        if col.startswith(("Home_Prev5", "Away_Prev5",
                           "Home_PrevSeasons", "Away_PrevSeasons"))
    ],
    "Head-to-Head Records": [
        col for col in final_d1.columns if col.startswith("H2H_")
    ],
    "Rest Days": ["Home_RestDays", "Away_RestDays"]
}

print("FINAL FEATURE ENGINEERING SUMMARY")
print("=" * 50)

print(f"Bundesliga 1: {final_d1.shape[0]} matches, {final_d1.shape[1]} columns")
print(f"Bundesliga 2: {final_d2.shape[0]} matches, {final_d2.shape[1]} columns")

print("\nENGINEERED FEATURES")

for category, features in feature_groups.items():
    print(f"\n{category} ({len(features)} features)")
    for feature in features:
        print(" -", feature)

# Display example matches with important engineered features
sample_cols = [
    "Season", "Date", "HomeTeam", "AwayTeam",
    "Home_Last5Points", "Away_Last5Points",
    "Home_Last5AvgGoalsFor", "Away_Last5AvgGoalsFor",
    "Home_Prev5SeasonAvgPPG", "Away_Prev5SeasonAvgPPG",
    "H2H_HomeWins", "H2H_AwayWins", "H2H_Draws",
    "Home_RestDays", "Away_RestDays", "FTR"
]

print("\nSAMPLE BUNDESLIGA 1 MATCHES")
display(final_d1[sample_cols].tail(5))

print("\nSAMPLE BUNDESLIGA 2 MATCHES")
display(final_d2[sample_cols].tail(5))

In [ ]:
# Compare the number of matches in each season across both divisions.
# Identify which season explains the 18-match difference.

season_counts = (
    df.groupby(["Season", "Division"])
      .size()
      .unstack()
)

season_counts["Difference_D2_minus_D1"] = (
    season_counts["D2"] - season_counts["D1"]
)

display(season_counts)

In [ ]:
# Export final cleaned datasets to CSV

final_d1.to_csv("Bundesliga1_Cleaned.csv", index=False)
final_d2.to_csv("Bundesliga2_Cleaned.csv", index=False)

print("Bundesliga 1 exported:", final_d1.shape)
print("Bundesliga 2 exported:", final_d2.shape)
print("Both CSV files saved successfully!")